# Continuous CPS — scikit-learn and LightGBM

Residual CPS with CDF, PPF, quantiles, coverage, and Newsvendor/capacity decisions. For LightGBM: `pip install lightgbm`.

In [8]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
import sys
import os
sys.path.append(os.path.abspath("../.."))
from tinyconformal.distribution import ContinuousConformalPredictiveSystem
from tinyconformal.utils import NewsvendorSolver

rng = np.random.default_rng(42)
X = rng.uniform(0, 10, size=(3000, 1))
y = 20 + 3 * X[:, 0] + rng.normal(0, 1 + 0.4 * X[:, 0])
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, random_state=42)
X_cal, X_test, y_cal, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

In [9]:
models = {
    "RandomForest": RandomForestRegressor(n_estimators=250, min_samples_leaf=8, random_state=42, n_jobs=-1),
    "LightGBM": LGBMRegressor(n_estimators=250, learning_rate=0.04, random_state=42, verbosity=-1),
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    cps = ContinuousConformalPredictiveSystem(model).fit(X_cal, y_cal)
    distribution = cps.predict_distribution(X_test)
    results[name] = (cps, distribution)
    display(distribution.evaluate(y_test))

,coverage,empirical_coverage,mean_width,winkler_score
0,0.50,0.506667,3.906938,8.053034
1,0.80,0.823333,8.584753,11.796050
2,0.90,0.896667,10.943468,14.191249
3,0.95,0.950000,13.570104,16.420729


,coverage,empirical_coverage,mean_width,winkler_score
0,0.50,0.526667,3.955622,8.100387
1,0.80,0.810000,8.364707,11.903579
2,0.90,0.910000,11.281148,14.497283
3,0.95,0.951667,13.549253,16.855302


## CDF, PPF, and quantile predictions

In [26]:
name = "RandomForest"
distribution = results[name][1]
quantile_levels = np.array([0.1, 0.5, 0.9])
quantile_matrix = np.broadcast_to(quantile_levels, (len(distribution), len(quantile_levels)))
quantile_predictions = distribution.ppf(quantile_matrix)

summary = pd.DataFrame({
    "y": y_test[:10],
    "cdf_at_y": distribution.cdf(y_test)[:10],
    "q10": quantile_predictions[:10, 0],
    "q50": quantile_predictions[:10, 1],
    "q90": quantile_predictions[:10, 2],
})
summary

,y,cdf_at_y,q10,q50,q90
0,49.222644,0.833611,41.756098,46.497900,50.340851
1,47.534625,0.963394,36.915436,41.657238,45.500190
2,41.568889,0.292845,38.649844,43.391646,47.234597
3,32.468934,0.898502,23.896383,28.638185,32.481137
4,39.612872,0.239601,37.142226,41.884028,45.726979
5,29.558778,0.645591,23.999878,28.741680,32.584631
6,38.664349,0.427621,34.570691,39.312493,43.155444
7,38.387074,0.722130,32.203066,36.944868,40.787819
8,28.941104,0.178037,27.401012,32.142815,35.985766
9,48.635388,0.928453,39.518462,44.260264,48.103215


## Solver

For continuous outcomes, the solver can represent optimal capacity. The row order must match the distribution.

In [28]:
decision_frame = pd.DataFrame({
    "unique_id": np.arange(len(y_test)).astype(str),
    "ds": pd.Timestamp("2026-01-01"),
    "shortage_cost": 9.0,
    "excess_cost": 1.0,
})
solver_result = NewsvendorSolver.optimize_distribution(
    decision_frame,
    distribution,
    underage_cost="shortage_cost",
    overage_cost="excess_cost",
)
solver_result.head()

,unique_id,ds,shortage_cost,excess_cost,critical_ratio,y_optimal
0,0,2026-01-01,9.0,1.0,0.9,50.340851
1,1,2026-01-01,9.0,1.0,0.9,45.500190
2,2,2026-01-01,9.0,1.0,0.9,47.234597
3,3,2026-01-01,9.0,1.0,0.9,32.481137
4,4,2026-01-01,9.0,1.0,0.9,45.726979
